# sw02 - solid model with embedded bars

Specimen **SW02** (Pizarro Pohl, Kovarbasic & Stojadinovic, ETH Zurich; JSE 2025, 151(4) 04025011): real-scale
squat wall 3000 x 2550 x 200, 1800 kN axial. Built like `RW2_Wallace/rw2_solid_ladruno_embedment.ipynb`: **hexahedra**
(`stdBrick`) and **discrete bars** embedded in the concrete (`LadrunoEmbeddedRebar`, perfect bond, `CorotTruss`
bars), specimen data (same values as `sw02_pizarro_specimen.ipynb`). Units: **N, mm, MPa**.

| item | here | source |
|---|---|---|
| wall | 3000 x 2550 x 200, no boundary element | JSE Fig. 1; thesis sec. 3.1.2 |
| foundation | 4000 x 500 x 200, one volume, elastic, E x (1650 / 200) = in-plane stiffness of the 1650 mm wide block | JSE Fig. 1; E baseline |
| loading beam | 3000 x 500 x 200, elastic, E x (900 / 200); its self weight kept at the real 900 mm width | JSE Fig. 1; E baseline |
| concrete | `ASDConcrete3D` 9P: fc' 28.5, Ec 32145; ft and ecp from EC2 | JSE Table 2 |
| vertical bars | 18 phi10 per face @ 150 (mark 10) + 1 phi16 per face at each end (mark 12) | drawing SW02.pdf |
| stirrups | closed phi10 @ 150 around the whole section (mark 11) | drawing SW02.pdf |
| steel | `Hysteretic`: phi10 523.65 / 569 at 2 % / 596.19 at 5.43 %; phi16 523.52 / 578 / 612.33 at 6.86 %; pinching 0.5 / 0.8 | JSE Fig. 3, coupons; pinching baseline |
| protocol | 46 measured peaks of the test, 0.25 mm increments | `SW02_Data.csv` |

Windows: runs with the Ladruno `OpenSeesMP` in **one process**. The installed apeGmsh cannot
export embedded-bar ties under partitioned MPI. Model inputs and loading history are unchanged.
Use the `C:\venv\opensees_venv\Scripts\python.exe` kernel. Evidence for every value: `context/`.

In [ ]:
import math
import os
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from apeGmsh import apeGmsh, Results
from apeGmsh.opensees import apeSees
from apeGmsh.rebar import Cage

notebook_dir = Path(r'D:\OneDrive\Escritorio\AHORA SI\Notebooks\SW02_Pizarro')
opensees_mp = Path(r'C:\OpenSees-Solvers\opensees-bin\OpenSeesMP.exe')
launcher = Path(r'C:\OpenSees-Solvers\openseesmp.bat')
nparts = 1  # installed apeGmsh cannot export embedded bars under partitioned MPI
run_dir = notebook_dir / 'sw02_pizarro_solid_ladruno_embedment'
exp_file = notebook_dir / 'SW02_Data.csv'  # ';' separated; X_Pos mm, X_Force kN

for required in (opensees_mp, launcher):
    if not required.is_file():
        raise FileNotFoundError(required)
run_dir.mkdir(parents=True, exist_ok=True)
print('Python:', sys.executable)
print('experiment:', exp_file)
print('run folder:', run_dir)
if not exp_file.is_file():
    print(f'Experimental file missing: {exp_file}. Model results remain available without the experimental overlay.')


## inputs (N, mm, MPa)

In [ ]:
# geometry, paper Fig. 1 and drawing SW02.pdf
lw = 3000.0          # wall length
hw = 2550.0          # wall height (foundation top to loading beam = control point)
tw = 200.0           # wall thickness (y direction, centred at y = 0)
max_size = 200.0     # maximum finite element size (same as the specimen shell models)
brick = 'ssp'        # LadrunoBrick formulation: std (= stdBrick) | bbar | ssp (stabilized one point, no bending locking) | eas
bar_refine = 4       # bar mesh = max_size / bar_refine (bars 4 times finer than the concrete)
base_overhang = 500.0
base_depth = 500.0
beam_h = 500.0       # loading beam height
record_all = True    # True: every concrete node (viewer, isometric views); False: loaded nodes only (calibration runs)

# concrete, paper Table 2 (SW02 wall: fc' 28.5, Ec 32145); tension not measured. Same as sw02_pizarro_specimen
ec = 32145.0
nu = 0.2
fcp = 28.5
ft = 0.3 * fcp ** (2 / 3)            # EC2 fctm
ecp = 0.7 * fcp ** 0.31 / 1000.0     # EC2 eps_c1
gt = 0.073 * fcp ** 0.18             # n/mm, fib model code 2010
fc0, fcr, gc = 5.0, 1.0, 33.0        # not reported: baseline values
pscale_t = pscale_c = 0.3

# foundation and loading beam: elastic (E not measured: baseline 33000), out-of-plane widths 1650 / 900
base_e = 33000.0 * 1650.0 / tw
beam_e = 33000.0 * 900.0 / tw
beam_t = 900.0

# bars, drawing marks 10-12; paper Fig. 3 (0.2 % offset fy) and coupons (mean stress at 2 %)
db10, db16 = 10.0, 16.0
fy10, es10, s2_10, fu10, eu10 = 523.65, 198970.0, 569.0, 596.19, 0.0543
fy16, es16, s2_16, fu16, eu16 = 523.52, 196490.0, 578.0, 612.33, 0.0686
pinch_x, pinch_y = 0.5, 0.8          # baseline (stko 02_DP)
hoop_edge = 25.0                 # stirrup axis to the face and to the wall end (hoop 295 x 16 cm)
edge10 = hoop_edge + (db10 + db10) / 2   # 35: phi10 vertical axis to the face, inside the stirrup
edge16 = hoop_edge + (db10 + db16) / 2   # 38: phi16 corner bar axis to the face and the wall end
v_n, v_step = 18, 150.0          # 18 phi10 vertical per face @ 150, centred
hoop_step = 150.0                # stirrups @ 150
anchor = 50.0                    # vertical bars stop 50 mm from the foundation bottom and the beam top
k_perfect = 1.0e8                # perfect-bond axial penalty of LadrunoEmbeddedRebar

# loads
p_axial = 1800000.0              # n, paper Table 1 (ALR 10.5 %); median Z_Force -1800 kN
gamma_c = 2.4e-9 * 9810.0        # n/mm3, self weight of the wall and the loading beam

print(f'axial {p_axial / 1e3:.0f} kn = {p_axial / (fcp * lw * tw):.3f} f\'c Ag | ft {ft:.2f} | ecp {ecp:.5f}')
print(f'steel phi10 {fy10} / {s2_10} / {fu10} | phi16 {fy16} / {s2_16} / {fu16} mpa | foundation e {base_e:.0f} | beam e {beam_e:.0f} mpa')

## geometry

Same x-z layout as `sw02_pizarro`, built on the face y = -tw/2 and then extruded through the thickness:
one foundation rectangle (one physical group, one volume), the wall and the loading beam. `fragment` cuts the
foundation top edge under the wall and, with two points, its bottom edge at the same x, so the foundation
meshes as one structured patch. `extrude` with layers and `recombine` gives hexahedra.

| physical group | volumes | use |
|---|---|---|
| `Wall` | wall | concrete |
| `Base_Block` | foundation | elastic |
| `Top_Beam` | loading beam | elastic, imposed displacement on all its nodes, base shear |
| `Concrete` | all volumes | host of the embedded bars (they run into the foundation and the beam) |
| `Fixed` | foundation bottom, both sides, exposed top | supports |
| `Top` | beam top face | axial load |

In [ ]:
g = apeGmsh(model_name='sw02_pizarro_solid_ladruno_embedment', verbose=False)
g.begin()

x0, x1 = -base_overhang, lw + base_overhang
y0 = -tw / 2                                 # extrusion face; the wall spans y -tw/2 .. tw/2

g.model.geometry.add_rectangle(x0, y0, -base_depth, x1 - x0, base_depth, plane='xz', label='base_block')
g.model.geometry.add_rectangle(0.0, y0, 0.0, lw, hw, plane='xz', label='wall')
g.model.geometry.add_rectangle(0.0, y0, hw, lw, beam_h, plane='xz', label='top_beam')
bottom_pts = [g.model.geometry.add_point(x, y0, -base_depth) for x in (0.0, lw)]
g.model.boolean.fragment(['base_block'], ['wall', 'top_beam'] + [(0, p) for p in bottom_pts], dim=2)

ny = math.ceil(tw / max_size - 1e-9) + 1       # nodes through the thickness
g.model.transforms.extrude(g.model.select(dim=2).tags(), 0.0, tw, 0.0, num_elements=[ny - 1], recombine=True)

tol = 1e-3
y1 = tw / 2 + tol
z_top = hw + beam_h

def vols_in(lo, hi):
    return g.model.select(dim=3).in_box(lo, hi).tags()

def faces_in(lo, hi):
    return g.model.select(dim=2).in_box(lo, hi).tags()

g.physical.add_volume(vols_in((x0 - tol, y0 - tol, -base_depth - tol), (x1 + tol, y1, tol)), name='Base_Block')
g.physical.add_volume(vols_in((-tol, y0 - tol, -tol), (lw + tol, y1, hw + tol)), name='Wall')
g.physical.add_volume(vols_in((-tol, y0 - tol, hw - tol), (lw + tol, y1, z_top + tol)), name='Top_Beam')
g.physical.add_volume(g.model.select(dim=3).tags(), name='Concrete')

fixed = (faces_in((x0 - tol, y0 - tol, -base_depth - tol), (x1 + tol, y1, -base_depth + tol))   # base bottom
         + faces_in((x0 - tol, y0 - tol, -base_depth - tol), (x0 + tol, y1, tol))                # left side
         + faces_in((x1 - tol, y0 - tol, -base_depth - tol), (x1 + tol, y1, tol))                # right side
         + faces_in((x0 - tol, y0 - tol, -tol), (tol, y1, tol))                                  # exposed top, left
         + faces_in((lw - tol, y0 - tol, -tol), (x1 + tol, y1, tol)))                            # exposed top, right
g.physical.add_surface(fixed, name='Fixed')
g.physical.add_surface(faces_in((-tol, y0 - tol, z_top - tol), (lw + tol, y1, z_top + tol)), name='Top')

with g.loads.case('selfweight'):
    g.loads.volume('Wall', force_per_volume=(0.0, 0.0, -gamma_c))
    g.loads.volume('Top_Beam', force_per_volume=(0.0, 0.0, -gamma_c * beam_t / tw))   # weight of the 900 mm beam

with g.loads.case('axial'):
    g.loads.surface.traction('Top', vector=(0.0, 0.0, -p_axial / (lw * tw)))

## reinforcement (drawing SW02.pdf, marks 10-12)

Bar centres (x from the left wall end, y from the wall mid-plane). Covers are not dimensioned; the stirrup
(295 x 16 cm, outer) gives its axis at 25 mm from the faces and the wall ends.

- **phi10 vertical** (mark 10): 18 per face @ 150 centred on the wall, x = 225 ... 2775, y = +-65.
- **phi16 vertical** (mark 12): 1 per face at each end, x = 38 and 2962, y = +-62 (stirrup corners).
- **stirrups** (mark 11): closed phi10 rectangles x 25 ... 2975, y = +-75, @ 150 from z = 75.

Vertical bars run from 50 mm above the foundation bottom to 50 mm below the beam top (the L ends and the
lap bars of marks 13-14 in the top 1050 / 1500 mm are not modelled). All bars: `CorotTruss` on their own
line mesh, tied to the hexahedra by `LadrunoEmbeddedRebar` with perfect bond.

In [ ]:
v_x = [lw / 2 + (j - (v_n - 1) / 2) * v_step for j in range(v_n)]   # 225 ... 2775
z_bot, z_end = -base_depth + anchor, z_top - anchor

bars = []

def add_bar(points, db, material, role):
    bars.append(g.rebar.bar(points, db=db, material=material, role=role, name=f'{role}_{len(bars)}'))

for s in (-1, 1):
    for x in v_x:
        add_bar([(x, s * (tw / 2 - edge10), z_bot), (x, s * (tw / 2 - edge10), z_end)], db10, 'steel_10', 'vertical')
    for x in (edge16, lw - edge16):
        add_bar([(x, s * (tw / 2 - edge16), z_bot), (x, s * (tw / 2 - edge16), z_end)], db16, 'steel_16', 'vertical')

a, b, yh = hoop_edge, lw - hoop_edge, tw / 2 - hoop_edge
z = hoop_step / 2
while z < hw:
    add_bar([(a, -yh, z), (b, -yh, z), (b, yh, z), (a, yh, z), (a, -yh, z)], db10, 'steel_10', 'tie')
    z += hoop_step

g.rebar.place(Cage(bars=tuple(bars)), into='Concrete', coupling='embedded', perfect=k_perfect,
              emit_elements=True)

n_role = {r: sum(b.role == r for b in bars) for r in ('vertical', 'tie')}
print('bars:', n_role)

## mesh

Structured hexahedra. On the x-z face each piece gets `n = ceil(length / max_size) + 1` nodes per side
(same rule as `rw2_stko`), the base is one transfinite patch with its four corners given, and the
extrusion puts `ny - 1` layers through the thickness. The bars are meshed on their own at
`max_size / bar_refine` (4 times finer than the concrete). The Windows version keeps the mesh unpartitioned (`nparts = 1`) for embedded-bar export.

In [ ]:
def n_nodes(length):
    return math.ceil(length / max_size - 1e-9) + 1

nx_wall, nx_over = n_nodes(lw), n_nodes(base_overhang)
nz_wall, nz_base, nz_beam = n_nodes(hw), n_nodes(base_depth), n_nodes(beam_h)
print(f'element size: wall {lw / (nx_wall - 1):.1f} | thickness {tw / (ny - 1):.1f} | '
      f'wall height {hw / (nz_wall - 1):.1f} | beam {beam_h / (nz_beam - 1):.1f} | '
      f'overhang {base_overhang / (nx_over - 1):.1f} | base depth {base_depth / (nz_base - 1):.1f} mm')

g.mesh.recipe.structured('wall', n=(nx_wall, nz_wall), generate=False)
g.mesh.recipe.structured('top_beam', n=(nx_wall, nz_beam), generate=False)

def curves_in(lo, hi):
    return g.model.select(dim=1).in_box(lo, hi).tags()

def point_at(x, z):
    return g.model.select(dim=0).in_box((x - tol, y0 - tol, z - tol), (x + tol, y0 + tol, z + tol)).tags()[0]

spans = [(x0, 0.0, nx_over), (0.0, lw, nx_wall), (lw, x1, nx_over)]
for z in (0.0, -base_depth):                       # foundation top and bottom edges, same divisions as the wall
    for a, b, n in spans:
        for c in curves_in((a - tol, y0 - tol, z - tol), (b + tol, y0 + tol, z + tol)):
            g.mesh.structured.set_transfinite_curve(c, n)
for x in (x0, x1):                                  # foundation sides
    for c in curves_in((x - tol, y0 - tol, -base_depth - tol), (x + tol, y0 + tol, tol)):
        g.mesh.structured.set_transfinite_curve(c, nz_base)
corners = [point_at(x0, -base_depth), point_at(x1, -base_depth), point_at(x1, 0.0), point_at(x0, 0.0)]
g.mesh.structured.set_transfinite_surface('base_block', corners=corners)
g.mesh.structured.set_recombine('base_block')
g.mesh.sizing.set_global_size(max_size / bar_refine)   # bar line mesh (the volumes are transfinite)

g.mesh.generation.generate(dim=3)
# Embedded-rebar MPI export is unsupported by the installed apeGmsh.
if nparts != 1:
    raise ValueError('This apeGmsh version requires nparts=1 for embedded bars')
fem = g.mesh.queries.get_fem_data(dim=None)

top_ids = [int(n) for n in fem.nodes.select(pg='Top_Beam').ids]   # imposed displacement, base shear
print(f'mesh : {fem.info.n_nodes} nodes | {fem.info.n_elems} elems | {nparts} partitions | types {fem.info.types}')
print(f'beam nodes {len(top_ids)} | fixed nodes {len(fem.nodes.select(pg="Fixed").ids)} | '
      f'embedded ties {len(g.reinforce.reinforce_records)}')

## section cut

Set `z_cut` and run: plan view of the mesh at that height. Grey rectangles are the hexahedra cut by the
plane; dots are bars crossing the plane (vertical bars, drawn at their diameter); red lines are stirrups
lying in the plane within `dz`. Stirrups are at z = 75 + 150 k.

In [ ]:
concrete_colors = {'Wall': '0.8', 'Base_Block': '0.95', 'Top_Beam': '0.95'}

In [ ]:
# section cut: concrete elements and bars at height z_cut
z_cut = 1275.0     # mm; 1275 has a stirrup (75 + 8 x 150)
dz = 5.0           # bars lying in the plane within +-dz are drawn as lines

xyz = dict(zip(fem.nodes.ids, fem.nodes.coords))

fig, ax = plt.subplots(figsize=(14, 3))
for grp, color in concrete_colors.items():
    for conn in fem.elements.select(pg=grp, element_type='hex8').connectivity:
        p = np.array([xyz[n] for n in conn])
        if p[:, 2].min() <= z_cut <= p[:, 2].max():
            ax.add_patch(plt.Rectangle((p[:, 0].min(), p[:, 1].min()), np.ptp(p[:, 0]), np.ptp(p[:, 1]),
                                       facecolor=color, edgecolor='0.4', lw=0.4))

bar_color = {'vertical': 'k', 'horizontal': 'tab:blue', 'tie': 'tab:red'}
for m in g.rebar.placements[0].members:
    for n1, n2 in fem.elements.select(pg=m.pg, element_type='line2').connectivity:
        a, b = np.array(xyz[n1]), np.array(xyz[n2])
        if abs(a[2] - z_cut) <= dz and abs(b[2] - z_cut) <= dz:              # bar in the cut plane
            ax.plot([a[0], b[0]], [a[1], b[1]], color=bar_color[m.role], lw=m.diameter / 3)
        elif min(a[2], b[2]) <= z_cut < max(a[2], b[2]):                     # bar crossing the cut
            t = (z_cut - a[2]) / (b[2] - a[2])
            p = a + t * (b - a)
            ax.add_patch(plt.Circle((p[0], p[1]), m.diameter / 2, color=bar_color[m.role]))

ax.set_aspect('equal')
ax.autoscale_view()
ax.set_xlabel('x (mm)')
ax.set_ylabel('y (mm)')
ax.set_title(f'section at z = {z_cut:.0f} mm  (black: vertical bars, blue: horizontal bars, red: hoops)')
plt.show()

## mesh viewer

In [ ]:
g.mesh.viewer()

## opensees model

In [ ]:
ops = apeSees(fem)
ops.model(ndm=3, ndf=3)

def concrete(fcp, ecp, fc0, fcr, gc):
    # Current API: reproduce the same STKO Concrete (9P) law and IMPL-EX alpha=1.
    from apeGmsh.interop.stko.translate_props import asdconcrete_9p
    from apeGmsh.opensees.material.nd import ASDConcrete3D

    law = asdconcrete_9p(dict(
        E=ec, ft=ft, fc0=fc0, fcp=fcp, fcr=fcr, ecp=ecp, Gt=gt, Gc=gc,
        **{'PScale Tension': pscale_t, 'PScale Compression': pscale_c}))
    # The current material validates damage >= 0; suppress only round-off.
    for key in ('Td', 'Cd'):
        if min(law[key]) < -1e-12:
            raise ValueError(f'Invalid negative damage in {key}')
        law[key] = tuple(max(0.0, value) for value in law[key])
    return ops.register(ASDConcrete3D(E=ec, v=nu, implex=True, **law))

conc = concrete(fcp, ecp, fc0, fcr, gc)
base_mat = ops.nDMaterial.ElasticIsotropic(E=base_e, nu=nu)
beam_mat = ops.nDMaterial.ElasticIsotropic(E=beam_e, nu=nu)

# bar materials: symmetric Hysteretic (yield, stress at 2 %, fu at eu), by the names used in g.rebar.bar
def hysteretic(fy, e_s, s2, fu, eu, name):
    ey = fy / e_s
    assert ey < 0.02 < eu and fy < s2 < fu, (fy, s2, fu, eu)
    return ops.uniaxialMaterial.Hysteretic(
        s1p=fy, e1p=ey, s2p=s2, e2p=0.02, s3p=fu, e3p=eu,
        s1n=-fy, e1n=-ey, s2n=-s2, e2n=-0.02, s3n=-fu, e3n=-eu,
        pinch_x=pinch_x, pinch_y=pinch_y, damage1=0.0, damage2=0.0, name=name)

hysteretic(fy10, es10, s2_10, fu10, eu10, 'steel_10')   # phi10: vertical bars and stirrups
hysteretic(fy16, es16, s2_16, fu16, eu16, 'steel_16')   # phi16 corner bars

ops.element.LadrunoBrick(pg='Wall', material=conc, formulation=brick)
ops.element.LadrunoBrick(pg='Base_Block', material=base_mat, formulation=brick)
ops.element.LadrunoBrick(pg='Top_Beam', material=beam_mat, formulation=brick)

ops.fix(pg='Fixed', dofs=(1, 1, 1))

## loading protocol (measured peaks of the test, same as sw02_pizarro_specimen)

In [ ]:
peaks = [0.6486, -0.4044, 0.6443, -0.4025, 1.3474, -1.0105, 1.3613, -1.0114,
         2.2989, -2.194, 2.3031, -2.197, 3.6039, -3.5094, 3.6457, -3.5118,
         5.0083, -4.9114, 5.0383, -4.9292, 7.7495, -7.6727, 7.785, -7.6852,
         10.4971, -10.4672, 10.5698, -10.4868, 13.4083, -13.3437, 13.4769, -13.3795,
         16.269, -16.2348, 16.3612, -16.2576, 19.1819, -19.146, 19.2955, -19.2014,
         22.1707, -22.1389, 22.3232, -22.2426, 25.3234, -25.9832]   # mm
disp_values = [0.0] + peaks
travel = np.cumsum(np.abs(np.diff(disp_values)))       # mm travelled up to each peak
time_values = [0.0] + [float(t) for t in travel / travel[-1]]   # pseudo-time proportional to travel
d_step = 0.25                                      # mm, displacement increment
n_cyclic = math.ceil(travel[-1] / d_step)
# hold the last peak a little past t = 1: the accumulated pseudo-time of the last increment can land a hair
# beyond 1.0, where the Path series returns 0 (a jump back to zero displacement)
disp_values.append(peaks[-1])
time_values.append(1.0 + 1.0 / n_cyclic)
print(f'{len(peaks)} peaks, {min(peaks):.2f} / {max(peaks):.2f} mm | travel {travel[-1]:.1f} mm | '
      f'{n_cyclic} increments of {travel[-1] / n_cyclic:.3f} mm')

## stages and deck

1. `gravity`: self weight of the wall and the beam + 1800 kN axial as a traction on the beam top, 10
   increments, then held.
2. `cyclic`: `sp` ux on every loading-beam node following the path (as `sw02_pizarro`).

Windows single-process export: `RCM` numberer and `UmfPack` direct solver. One `.ladruno` recorder (displacement +
reaction of every node) in the cyclic stage. Base shear = sum of the x reactions of the beam nodes.

`record_all = True` records every concrete node (results viewer, isometric views); `False` records only
the loaded nodes (small files, enough for the base shear curve; used in the calibration runs).

In [ ]:
rec = ops.recorder.Ladruno(file='sw02_pizarro_solid_ladruno_embedment.ladruno', nodal_responses=('displacement', 'reactionForce'),
                           nodes_pg='Concrete' if record_all else 'Top_Beam')

n_gravity = 10
dlam_gravity = 1.0 / n_gravity

with ops.stage(name='gravity') as s:
    with s.pattern(series=ops.timeSeries.Linear()) as p:
        p.from_model('selfweight')
        p.from_model('axial')
    s.analysis(test=ops.test.EnergyIncr(tol=1e-4, max_iter=10),
               algorithm=ops.algorithm.KrylovNewton(max_dim=10),
               integrator=ops.integrator.LoadControl(dlam=dlam_gravity),
               constraints=ops.constraints.Transformation(),
               numberer=ops.numberer.RCM(),
               system=ops.system.UmfPack(),
               analysis=ops.analysis.Static())
    s.run(n_increments=n_gravity, dt=dlam_gravity)

dlam_cyclic = 1.0 / n_cyclic

with ops.stage(name='cyclic') as s:
    with s.pattern(series=ops.timeSeries.Path(values=disp_values, time=time_values)) as p:
        p.sp(pg='Top_Beam', dof=1, value=1.0)
    s.recorder(rec)
    s.analysis(test=ops.test.NormDispIncr(tol=1e-4, max_iter=100),
               algorithm=ops.algorithm.KrylovNewton(max_dim=10),
               integrator=ops.integrator.LoadControl(dlam=dlam_cyclic),
               constraints=ops.constraints.Transformation(),
               numberer=ops.numberer.RCM(),
               system=ops.system.UmfPack(),
               analysis=ops.analysis.Static())
    # on a failed increment escalate ModifiedNewton(initial) -> KrylovNewton -> NewtonLineSearch
    s.run(n_increments=n_cyclic, dt=dlam_cyclic, strategy=ops.strategy.profile('penalty-stiff'))

ops.tcl(str(run_dir / 'sw02_pizarro_solid_ladruno_embedment.tcl'))
g.end()
print('deck:', run_dir / 'sw02_pizarro_solid_ladruno_embedment.tcl')

## run (Windows OpenSeesMP, one process)

The Windows launcher initializes the solver environment. The cell displays progress and a status
message every 30 seconds without new progress. On failure or interruption, only this run and its
child processes are stopped.


In [ ]:
deck_file = run_dir / 'sw02_pizarro_solid_ladruno_embedment.tcl'
if not deck_file.is_file():
    raise FileNotFoundError(f'Generate the deck first: {deck_file}')
if nparts != 1:
    raise ValueError('This apeGmsh version requires nparts=1 for embedded bars')

for f in run_dir.glob('sw02_pizarro_solid_ladruno_embedment*.ladruno'):   # clean HDF5 recorders on rerun
    f.unlink()

log_file = run_dir / 'sw02_pizarro_solid_ladruno_embedment_run.log'
failed = False
started = time.monotonic()
last_update = started
last_progress = ''
print(f'Starting OpenSees: {deck_file}', flush=True)
# With no second argument, this Windows launcher runs OpenSeesMP in one process.
# It also initializes Intel MPI, Tcl and the solver DLL search paths.
with log_file.open('w', encoding='utf-8') as log:
    proc = subprocess.Popen(
        [os.environ.get('COMSPEC', 'cmd.exe'), '/d', '/c', str(launcher), str(deck_file)],
        cwd=run_dir, stdout=log, stderr=subprocess.STDOUT,
        creationflags=subprocess.CREATE_NO_WINDOW)
    try:
        while proc.poll() is None:
            try:
                proc.wait(timeout=10)
            except subprocess.TimeoutExpired:
                pass
            text = log_file.read_text(encoding='utf-8', errors='replace')
            progress = [line for line in text.splitlines() if 'APEGMSH_PROGRESS' in line]
            elapsed = time.monotonic() - started
            if progress and progress[-1] != last_progress:
                last_progress = progress[-1]
                print(f'[{elapsed / 60:.1f} min] {last_progress}', flush=True)
                last_update = time.monotonic()
            elif time.monotonic() - last_update >= 30:
                print(f'[{elapsed / 60:.1f} min] OpenSees is running; '
                      f'last progress: {last_progress or "waiting for first increment"}', flush=True)
                last_update = time.monotonic()
            if any(token in text for token in ('FAILED', 'ERROR', 'FATAL', 'while executing')):
                failed = True
                break
    finally:
        if proc.poll() is None:   # this run only, including children; also handles kernel interrupts
            subprocess.run(['taskkill', '/PID', str(proc.pid), '/T', '/F'],
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                           creationflags=subprocess.CREATE_NO_WINDOW, check=False)
            proc.wait(timeout=30)

text = log_file.read_text(encoding='utf-8', errors='replace')
failed = failed or any(token in text for token in ('FAILED', 'ERROR', 'FATAL', 'while executing'))
print('return code:', proc.returncode, '| failed:', failed)
print([line for line in text.splitlines() if 'APEGMSH_PROGRESS' in line][-1:])
assert not failed and proc.returncode == 0, f'opensees run failed, see {log_file}'
assert f'APEGMSH_PROGRESS i={n_cyclic} n={n_cyclic} ' in text, f'Incomplete cyclic analysis, see {log_file}'


## results vs experiment

| source | file units | conversion |
|---|---|---|
| model (`.ladruno`, one file per rank, merged by `Results.from_ladruno`) | disp mm, reaction N | sum of the x reactions of the loaded nodes / 1000 |
| experiment (`SW02_Data.csv`, `;`) | `X_Pos` mm, `X_Force` kN | none |
The experimental overlay is shown only when `SW02_Data.csv` is present next to the notebook.
Without it, the numerical results and their viewer remain available.


In [ ]:
import h5py


def _has_nodes(path):
    with h5py.File(path, 'r') as f:
        return any(isinstance(g, h5py.Group) and 'MODEL/NODES' in g for g in f.values())


name = 'sw02_pizarro_solid_ladruno_embedment'
# a rank that owns none of the recorded nodes writes an empty file: read only the files with data
parts = [str(p) for p in sorted(run_dir.glob(f'{name}*.ladruno')) if _has_nodes(p)]
results = Results.from_ladruno(parts, merge_partitions=True)

disp = results.nodes.get(ids=top_ids, component='displacement_x').values[:, 0]                    # mm
shear = results.nodes.get(ids=top_ids, component='reaction_force_x').values.sum(axis=1) / 1000.0  # kn
n = min(len(disp), len(shear))
disp, shear = disp[:n], shear[:n]

experiment_available = exp_file.is_file()
if experiment_available:
    exp = pd.read_csv(exp_file, sep=';')
    exp_disp, exp_shear = exp['X_Pos'].to_numpy(float), exp['X_Force'].to_numpy(float)   # mm, kn
else:
    print(f'Experimental comparison unavailable; missing file: {exp_file}')

curves = [('solid', disp, shear)]
if experiment_available:
    curves.append(('experiment', exp_disp, exp_shear))
for label, d, v in curves:
    print(f'{label:10s}: {len(d):5d} pts | disp {d.min():7.1f} / {d.max():6.1f} mm | shear {v.min():8.1f} / {v.max():7.1f} kn')

fig, ax = plt.subplots(figsize=(8, 5))
if experiment_available:
    ax.plot(exp_disp, exp_shear, 'k--', lw=1.0, label='experiment')
ax.plot(disp, shear, lw=1.0, label=f'solid, embedded bars (max_size {max_size:.0f})')
ax.set_xlabel('top displacement (mm)')
ax.set_ylabel('base shear (kN)')
ax.set_title('SW02 solid - base shear vs top displacement')
ax.grid(True)
ax.legend()
plt.show()

## results viewer

In [ ]:
results.viewer()

## results only (run from here)

Stand-alone: imports, reads this notebook's deck and `.ladruno` files from the last run and plots against the experiment. No need to run the cells above.
Without `SW02_Data.csv`, this cell displays the numerical response only and reports the missing file.


In [ ]:
# results only: run this cell by itself (fresh kernel is fine): it reads the deck and the .ladruno of the last run
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from apeGmsh import Results
import h5py


def _has_nodes(path):
    with h5py.File(path, 'r') as f:
        return any(isinstance(g, h5py.Group) and 'MODEL/NODES' in g for g in f.values())


name = 'sw02_pizarro_solid_ladruno_embedment'
notebook_dir = Path(r'D:\OneDrive\Escritorio\AHORA SI\Notebooks\SW02_Pizarro')
run_dir = notebook_dir / name
exp_file = notebook_dir / 'SW02_Data.csv'   # ';', X_Pos mm, X_Force kN

deck = (run_dir / f'{name}.tcl').read_text(encoding='utf-8')
top_ids = sorted({int(n) for n in re.findall(r'^\s*sp (\d+) 1 ', deck, re.M)})   # nodes with the imposed displacement

# a rank that owns none of the recorded nodes writes an empty file: read only the files with data
parts = [str(p) for p in sorted(run_dir.glob(f'{name}*.ladruno')) if _has_nodes(p)]
results = Results.from_ladruno(parts, merge_partitions=True)

disp = results.nodes.get(ids=top_ids, component='displacement_x').values[:, 0]                    # mm
shear = results.nodes.get(ids=top_ids, component='reaction_force_x').values.sum(axis=1) / 1000.0  # kn
n = min(len(disp), len(shear))
disp, shear = disp[:n], shear[:n]

experiment_available = exp_file.is_file()
if experiment_available:
    exp = pd.read_csv(exp_file, sep=';')
    exp_disp, exp_shear = exp['X_Pos'].to_numpy(float), exp['X_Force'].to_numpy(float)   # mm, kn
else:
    print(f'Experimental comparison unavailable; missing file: {exp_file}')

curves = [(name, disp, shear)]
if experiment_available:
    curves.append(('experiment', exp_disp, exp_shear))
for label, d, v in curves:
    print(f'{label:20s}: {len(d):5d} pts | disp {d.min():7.1f} / {d.max():6.1f} mm | shear {v.min():8.1f} / {v.max():7.1f} kn')

fig, ax = plt.subplots(figsize=(8, 5))
if experiment_available:
    ax.plot(exp_disp, exp_shear, 'k--', lw=1.0, label='experiment')
ax.plot(disp, shear, lw=1.0, label=name)
ax.set_xlabel('top displacement (mm)')
ax.set_ylabel('base shear (kN)')
ax.set_title('SW02 - base shear vs top displacement')
ax.grid(True)
ax.legend()
plt.show()